
# Roxy notebook example: Global basic protein sequence descriptors

This notebook is a **reference implementation example** for the **global basic descriptor family** in Roxy.

It uses a tiny demo dataset and shows how to compute a broad set of **sequence-level global descriptors** from raw protein sequences, without embeddings, structures, docking, or external models.

## Covered descriptor groups

This notebook implements global/basic descriptors such as:

- sequence length
- amino-acid validity counts
- approximate molecular weight
- aromaticity
- aliphatic fraction and aliphatic index
- polar / nonpolar / charged residue fractions
- positive and negative residue fractions
- tiny / small / branched / sulfur / amide / hydroxyl residue fractions
- hydrophobicity summaries
- polarity summaries
- flexibility summaries
- helix / sheet / turn propensity summaries
- disorder-promoting and order-promoting fractions
- net charge approximation at pH 7
- FCR and NCPR
- acidic/basic ratios
- donors and acceptors per residue
- Shannon entropy of composition
- sequence complexity proxies
- longest homopolymer run
- repeated dipeptide burden
- simple amphipathic proxy based on local hydropathy variation

The code is written as a **clear teaching example** so it can later be adapted into the real Roxy package.


In [1]:

import math
from collections import Counter
from itertools import groupby

import numpy as np
import pandas as pd



## Demo dataset

This is only a small synthetic/demo dataset to illustrate implementation patterns.


In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "demo_1",
            "demo_2",
            "demo_3",
            "demo_4",
            "demo_5",
            "demo_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,demo_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,demo_2,GGGGGGGGGGGGGGG,B
2,demo_3,KRRKRRKRRKRRDDDDEE,A
3,demo_4,ACDEFGHIKLMNPQRSTVWY,B
4,demo_5,PPPPGSSSSSTTTTNNQQQ,A
5,demo_6,MSTNPKPQRITLKDGNKVELV,B


## Amino-acid constants and scales

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)

AA_MOLECULAR_WEIGHT = {
    "A": 89.09, "C": 121.15, "D": 133.10, "E": 147.13, "F": 165.19,
    "G": 75.07, "H": 155.16, "I": 131.17, "K": 146.19, "L": 131.17,
    "M": 149.21, "N": 132.12, "P": 115.13, "Q": 146.15, "R": 174.20,
    "S": 105.09, "T": 119.12, "V": 117.15, "W": 204.23, "Y": 181.19,
}

HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

POLARITY = {
    "A": 8.1, "C": 5.5, "D": 13.0, "E": 12.3, "F": 5.2,
    "G": 9.0, "H": 10.4, "I": 5.2, "K": 11.3, "L": 4.9,
    "M": 5.7, "N": 11.6, "P": 8.0, "Q": 10.5, "R": 10.5,
    "S": 9.2, "T": 8.6, "V": 5.9, "W": 5.4, "Y": 6.2,
}

FLEXIBILITY = {
    "A": 0.357, "C": 0.346, "D": 0.511, "E": 0.497, "F": 0.314,
    "G": 0.544, "H": 0.323, "I": 0.462, "K": 0.466, "L": 0.365,
    "M": 0.295, "N": 0.463, "P": 0.509, "Q": 0.493, "R": 0.529,
    "S": 0.507, "T": 0.444, "V": 0.386, "W": 0.305, "Y": 0.420,
}

HELIX_PROP = {
    "A": 1.45, "C": 0.77, "D": 1.01, "E": 1.53, "F": 1.12,
    "G": 0.53, "H": 1.24, "I": 1.00, "K": 1.07, "L": 1.34,
    "M": 1.20, "N": 0.73, "P": 0.59, "Q": 1.17, "R": 0.79,
    "S": 0.79, "T": 0.82, "V": 1.14, "W": 1.14, "Y": 0.61,
}

SHEET_PROP = {
    "A": 0.97, "C": 1.30, "D": 0.54, "E": 0.37, "F": 1.28,
    "G": 0.81, "H": 0.71, "I": 1.60, "K": 0.74, "L": 1.22,
    "M": 1.67, "N": 0.65, "P": 0.62, "Q": 1.23, "R": 0.90,
    "S": 0.72, "T": 1.20, "V": 1.65, "W": 1.19, "Y": 1.29,
}

TURN_PROP = {
    "A": 0.66, "C": 1.19, "D": 1.46, "E": 0.74, "F": 0.60,
    "G": 1.56, "H": 0.95, "I": 0.47, "K": 1.01, "L": 0.59,
    "M": 0.60, "N": 1.56, "P": 1.52, "Q": 0.98, "R": 0.95,
    "S": 1.43, "T": 0.96, "V": 0.50, "W": 0.96, "Y": 1.14,
}

BOMAN_SCALE = {
    "A": 0.17, "C": 0.24, "D": -1.23, "E": -2.02, "F": 1.13,
    "G": 0.01, "H": -0.96, "I": 0.31, "K": -0.99, "L": 0.56,
    "M": 0.23, "N": -0.42, "P": -0.45, "Q": -0.58, "R": -1.01,
    "S": -0.13, "T": -0.14, "V": 0.07, "W": 1.85, "Y": 0.94,
}

P_KA_SIDECHAIN = {
    "C": 8.3,
    "D": 3.9,
    "E": 4.3,
    "H": 6.0,
    "K": 10.5,
    "R": 12.5,
    "Y": 10.1,
}
P_KA_N_TERM = 9.69
P_KA_C_TERM = 2.34

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "tiny": set("AGCS"),
    "small": set("AGCSTVPDN"),
    "branched": set("VILT"),
    "sulfur": set("CM"),
    "hydroxyl": set("STY"),
    "amide": set("NQ"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
    "acidic": set("DE"),
    "basic": set("KRH"),
}

HBOND_DONORS = {
    "A": 0, "C": 0, "D": 0, "E": 0, "F": 0,
    "G": 0, "H": 1, "I": 0, "K": 1, "L": 0,
    "M": 0, "N": 1, "P": 0, "Q": 1, "R": 1,
    "S": 1, "T": 1, "V": 0, "W": 1, "Y": 1,
}

HBOND_ACCEPTORS = {
    "A": 0, "C": 1, "D": 2, "E": 2, "F": 0,
    "G": 0, "H": 1, "I": 0, "K": 0, "L": 0,
    "M": 1, "N": 1, "P": 0, "Q": 1, "R": 0,
    "S": 1, "T": 1, "V": 0, "W": 0, "Y": 1,
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    seq = "".join([aa for aa in seq if aa in STANDARD_AA_SET])
    return seq


def fraction_from_group(seq: str, aa_group) -> float:
    if not seq:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def scale_values(seq: str, scale: dict) -> list:
    return [scale[aa] for aa in seq if aa in scale]


def safe_mean(values: list) -> float:
    return float(np.mean(values)) if values else np.nan


def safe_std(values: list) -> float:
    return float(np.std(values, ddof=0)) if values else np.nan


def shannon_entropy_from_counts(counts: Counter) -> float:
    total = sum(counts.values())
    if total == 0:
        return np.nan
    probs = np.array([c / total for c in counts.values()], dtype=float)
    return float(-(probs * np.log2(probs)).sum())


def linguistic_complexity(seq: str, k: int) -> float:
    if not seq or len(seq) < k or k < 1:
        return np.nan
    observed = len({seq[i:i+k] for i in range(len(seq) - k + 1)})
    possible = min(20**k, len(seq) - k + 1)
    return observed / possible if possible > 0 else np.nan


def longest_homopolymer_run(seq: str) -> int:
    if not seq:
        return 0
    return max(len(list(group)) for _, group in groupby(seq))


def repeated_dipeptide_fraction(seq: str) -> float:
    if len(seq) < 2:
        return np.nan
    kmers = [seq[i:i+2] for i in range(len(seq) - 1)]
    counts = Counter(kmers)
    repeated = sum(v for v in counts.values() if v > 1)
    return repeated / len(kmers)


def aliphatic_index(seq: str) -> float:
    if not seq:
        return np.nan
    counts = Counter(seq)
    a = counts["A"] / len(seq)
    v = counts["V"] / len(seq)
    i = counts["I"] / len(seq)
    l = counts["L"] / len(seq)
    return 100 * (a + 2.9 * v + 3.9 * (i + l))


def approximate_molecular_weight(seq: str) -> float:
    if not seq:
        return np.nan
    total = sum(AA_MOLECULAR_WEIGHT[aa] for aa in seq)
    return total - (len(seq) - 1) * 18.015


def residue_count(seq: str, aa_set) -> int:
    return sum(aa in aa_set for aa in seq)


def safe_ratio(num: float, den: float) -> float:
    if den == 0:
        return np.nan
    return num / den


def net_charge_at_ph(seq: str, ph: float = 7.0) -> float:
    if not seq:
        return np.nan

    counts = Counter(seq)

    n_term = 1 / (1 + 10 ** (ph - P_KA_N_TERM))
    k = counts["K"] * (1 / (1 + 10 ** (ph - P_KA_SIDECHAIN["K"])))
    r = counts["R"] * (1 / (1 + 10 ** (ph - P_KA_SIDECHAIN["R"])))
    h = counts["H"] * (1 / (1 + 10 ** (ph - P_KA_SIDECHAIN["H"])))

    c_term = 1 / (1 + 10 ** (P_KA_C_TERM - ph))
    d = counts["D"] * (1 / (1 + 10 ** (P_KA_SIDECHAIN["D"] - ph)))
    e = counts["E"] * (1 / (1 + 10 ** (P_KA_SIDECHAIN["E"] - ph)))
    c = counts["C"] * (1 / (1 + 10 ** (P_KA_SIDECHAIN["C"] - ph)))
    y = counts["Y"] * (1 / (1 + 10 ** (P_KA_SIDECHAIN["Y"] - ph)))

    positive_charge = n_term + k + r + h
    negative_charge = c_term + d + e + c + y
    return positive_charge - negative_charge


def local_hydropathy_amplitude(seq: str, window: int = 5) -> float:
    vals = scale_values(seq, HYDROPATHY)
    if len(vals) < window:
        return np.nan
    window_means = [np.mean(vals[i:i+window]) for i in range(len(vals) - window + 1)]
    return float(np.max(window_means) - np.min(window_means))


def sequence_global_basic_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    counts = Counter(seq)
    length = len(seq)

    if length == 0:
        return {
            "basic_length": 0,
            "basic_valid_residue_count": 0,
            "basic_unique_residue_count": 0,
        }

    hydropathy_vals = scale_values(seq, HYDROPATHY)
    polarity_vals = scale_values(seq, POLARITY)
    flexibility_vals = scale_values(seq, FLEXIBILITY)
    helix_vals = scale_values(seq, HELIX_PROP)
    sheet_vals = scale_values(seq, SHEET_PROP)
    turn_vals = scale_values(seq, TURN_PROP)
    boman_vals = scale_values(seq, BOMAN_SCALE)

    positive_count = residue_count(seq, AA_GROUPS["positive"])
    negative_count = residue_count(seq, AA_GROUPS["negative"])
    charged_count = residue_count(seq, AA_GROUPS["charged"])
    acidic_count = residue_count(seq, AA_GROUPS["acidic"])
    basic_count = residue_count(seq, AA_GROUPS["basic"])

    descriptors = {
        "basic_length": length,
        "basic_valid_residue_count": length,
        "basic_unique_residue_count": len(counts),
        "basic_molecular_weight": approximate_molecular_weight(seq),
        "basic_aromatic_fraction": fraction_from_group(seq, AA_GROUPS["aromatic"]),
        "basic_aliphatic_fraction": fraction_from_group(seq, AA_GROUPS["aliphatic"]),
        "basic_aliphatic_index": aliphatic_index(seq),
        "basic_polar_fraction": fraction_from_group(seq, AA_GROUPS["polar"]),
        "basic_nonpolar_fraction": fraction_from_group(seq, AA_GROUPS["nonpolar"]),
        "basic_positive_fraction": positive_count / length,
        "basic_negative_fraction": negative_count / length,
        "basic_charged_fraction": charged_count / length,
        "basic_tiny_fraction": fraction_from_group(seq, AA_GROUPS["tiny"]),
        "basic_small_fraction": fraction_from_group(seq, AA_GROUPS["small"]),
        "basic_branched_fraction": fraction_from_group(seq, AA_GROUPS["branched"]),
        "basic_sulfur_fraction": fraction_from_group(seq, AA_GROUPS["sulfur"]),
        "basic_hydroxyl_fraction": fraction_from_group(seq, AA_GROUPS["hydroxyl"]),
        "basic_amide_fraction": fraction_from_group(seq, AA_GROUPS["amide"]),
        "basic_hydrophobic_fraction": fraction_from_group(seq, AA_GROUPS["hydrophobic"]),
        "basic_hydrophilic_fraction": fraction_from_group(seq, AA_GROUPS["hydrophilic"]),
        "basic_disorder_promoting_fraction": fraction_from_group(seq, AA_GROUPS["disorder_promoting"]),
        "basic_order_promoting_fraction": fraction_from_group(seq, AA_GROUPS["order_promoting"]),
        "basic_hydropathy_mean": safe_mean(hydropathy_vals),
        "basic_hydropathy_std": safe_std(hydropathy_vals),
        "basic_polarity_mean": safe_mean(polarity_vals),
        "basic_polarity_std": safe_std(polarity_vals),
        "basic_flexibility_mean": safe_mean(flexibility_vals),
        "basic_flexibility_std": safe_std(flexibility_vals),
        "basic_helix_propensity_mean": safe_mean(helix_vals),
        "basic_sheet_propensity_mean": safe_mean(sheet_vals),
        "basic_turn_propensity_mean": safe_mean(turn_vals),
        "basic_boman_index_mean": safe_mean(boman_vals),
        "basic_net_charge_ph7": net_charge_at_ph(seq, ph=7.0),
        "basic_fcr": charged_count / length,
        "basic_ncpr": (positive_count - negative_count) / length,
        "basic_acidic_basic_ratio": safe_ratio(acidic_count, basic_count),
        "basic_basic_acidic_ratio": safe_ratio(basic_count, acidic_count),
        "basic_donors_per_residue": sum(HBOND_DONORS[aa] for aa in seq) / length,
        "basic_acceptors_per_residue": sum(HBOND_ACCEPTORS[aa] for aa in seq) / length,
        "basic_shannon_entropy": shannon_entropy_from_counts(counts),
        "basic_linguistic_complexity_k1": linguistic_complexity(seq, 1),
        "basic_linguistic_complexity_k2": linguistic_complexity(seq, 2),
        "basic_linguistic_complexity_k3": linguistic_complexity(seq, 3),
        "basic_longest_homopolymer_run": longest_homopolymer_run(seq),
        "basic_repeated_dipeptide_fraction": repeated_dipeptide_fraction(seq),
        "basic_local_hydropathy_amplitude_w5": local_hydropathy_amplitude(seq, window=5),
    }

    return descriptors


## Compute global basic descriptors for the demo dataset

In [5]:

df_global = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(sequence_global_basic_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_global


,sequence_id,sequence,label,basic_length,basic_valid_residue_count,basic_unique_residue_count,basic_molecular_weight,basic_aromatic_fraction,basic_aliphatic_fraction,basic_aliphatic_index,...,basic_basic_acidic_ratio,basic_donors_per_residue,basic_acceptors_per_residue,basic_shannon_entropy,basic_linguistic_complexity_k1,basic_linguistic_complexity_k2,basic_linguistic_complexity_k3,basic_longest_homopolymer_run,basic_repeated_dipeptide_fraction,basic_local_hydropathy_amplitude_w5
0,demo_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,13.0,2912.455,0.25,0.333333,93.333333,...,NaN,0.458333,0.291667,3.438722,0.650000,0.956522,1.000000,2.0,0.086957,4.52
1,demo_2,GGGGGGGGGGGGGGG,B,15.0,15.0,1.0,873.840,0.00,0.000000,0.000000,...,NaN,0.000000,0.000000,-0.000000,0.066667,0.071429,0.076923,15.0,1.000000,0.00
2,demo_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,4.0,2498.765,0.00,0.000000,0.000000,...,2.0,0.666667,0.666667,1.836592,0.222222,0.411765,0.500000,4.0,0.823529,0.88
3,demo_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,20.0,2395.725,0.20,0.250000,58.500000,...,1.5,0.450000,0.600000,4.321928,1.000000,1.000000,1.000000,1.0,0.000000,3.40
4,demo_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,6.0,1915.940,0.00,0.000000,0.000000,...,NaN,0.736842,0.736842,2.439268,0.315789,0.555556,0.764706,5.0,0.666667,2.78
5,demo_6,MSTNPKPQRITLKDGNKVELV,B,21.0,21.0,14.0,2368.770,0.00,0.285714,83.333333,...,2.0,0.476190,0.523810,3.689704,0.700000,1.000000,1.000000,1.0,0.000000,4.00


## Inspect descriptor columns

In [6]:

descriptor_cols = [c for c in df_global.columns if c.startswith("basic_")]
descriptor_cols


['basic_length',
 'basic_valid_residue_count',
 'basic_unique_residue_count',
 'basic_molecular_weight',
 'basic_aromatic_fraction',
 'basic_aliphatic_fraction',
 'basic_aliphatic_index',
 'basic_polar_fraction',
 'basic_nonpolar_fraction',
 'basic_positive_fraction',
 'basic_negative_fraction',
 'basic_charged_fraction',
 'basic_tiny_fraction',
 'basic_small_fraction',
 'basic_branched_fraction',
 'basic_sulfur_fraction',
 'basic_hydroxyl_fraction',
 'basic_amide_fraction',
 'basic_hydrophobic_fraction',
 'basic_hydrophilic_fraction',
 'basic_disorder_promoting_fraction',
 'basic_order_promoting_fraction',
 'basic_hydropathy_mean',
 'basic_hydropathy_std',
 'basic_polarity_mean',
 'basic_polarity_std',
 'basic_flexibility_mean',
 'basic_flexibility_std',
 'basic_helix_propensity_mean',
 'basic_sheet_propensity_mean',
 'basic_turn_propensity_mean',
 'basic_boman_index_mean',
 'basic_net_charge_ph7',
 'basic_fcr',
 'basic_ncpr',
 'basic_acidic_basic_ratio',
 'basic_basic_acidic_ratio',


In [7]:

df_global[["sequence_id"] + descriptor_cols].head()


,sequence_id,basic_length,basic_valid_residue_count,basic_unique_residue_count,basic_molecular_weight,basic_aromatic_fraction,basic_aliphatic_fraction,basic_aliphatic_index,basic_polar_fraction,basic_nonpolar_fraction,...,basic_basic_acidic_ratio,basic_donors_per_residue,basic_acceptors_per_residue,basic_shannon_entropy,basic_linguistic_complexity_k1,basic_linguistic_complexity_k2,basic_linguistic_complexity_k3,basic_longest_homopolymer_run,basic_repeated_dipeptide_fraction,basic_local_hydropathy_amplitude_w5
0,demo_1,24.0,24.0,13.0,2912.455,0.25,0.333333,93.333333,0.458333,0.541667,...,NaN,0.458333,0.291667,3.438722,0.650000,0.956522,1.000000,2.0,0.086957,4.52
1,demo_2,15.0,15.0,1.0,873.840,0.00,0.000000,0.000000,0.000000,1.000000,...,NaN,0.000000,0.000000,-0.000000,0.066667,0.071429,0.076923,15.0,1.000000,0.00
2,demo_3,18.0,18.0,4.0,2498.765,0.00,0.000000,0.000000,1.000000,0.000000,...,2.0,0.666667,0.666667,1.836592,0.222222,0.411765,0.500000,4.0,0.823529,0.88
3,demo_4,20.0,20.0,20.0,2395.725,0.20,0.250000,58.500000,0.600000,0.400000,...,1.5,0.450000,0.600000,4.321928,1.000000,1.000000,1.000000,1.0,0.000000,3.40
4,demo_5,19.0,19.0,6.0,1915.940,0.00,0.000000,0.000000,0.736842,0.263158,...,NaN,0.736842,0.736842,2.439268,0.315789,0.555556,0.764706,5.0,0.666667,2.78


## Quick sanity checks

In [8]:

assert "basic_length" in df_global.columns
assert "basic_molecular_weight" in df_global.columns
assert "basic_net_charge_ph7" in df_global.columns
assert "basic_shannon_entropy" in df_global.columns
assert df_global["basic_length"].min() > 0

print(f"Number of global/basic descriptors implemented: {len(descriptor_cols)}")


Number of global/basic descriptors implemented: 46



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move the scales and amino-acid groups into `roxy/core/constants.py`
- move helper functions into `roxy/sequence/utils.py`
- convert `sequence_global_basic_descriptors` into a class such as `BasicGlobalDescriptors`
- make `.transform()` accept lists, pandas Series, and DataFrames
- add unit tests for edge cases:
  - empty sequence
  - sequence with invalid characters
  - sequence of length 1
  - highly repetitive sequences
  - sequences with only charged residues



## Optional export

Uncomment the next cell if you want to save the descriptor table.


In [ ]:
# df_global.to_csv("demo_global_basic_descriptors.csv", index=False)
